# 04 — LLM Zero-Shot Baseline (T133)

Classifies a stratified 500-row sample of the held-out test split with
**Groq `llama-3.1-8b-instant`** via the OpenAI-compatible endpoint at
`https://api.groq.com/openai/v1`. No training, no fine-tuning — pure prompt
classification.

**Sampling deviation from spec**: the original T133 spec calls for the full
2,290 test rows. Groq free-tier rate-limits `llama-3.1-8b-instant` at
30 RPM, so 2,290 sequential calls would take ~76 minutes. We instead sample
100 rows per label (stratified, `random_state=42`) for a balanced 500-row
eval that finishes in ~18 minutes. T134's comparison will note the
sample-size asymmetry vs the full-test runs from T131/T132.

**Pricing** (Groq published rates, USD per 1M tokens):
* Input: $0.05  
* Output: $0.08

**Outputs**:
* `notebooks/results/llm_zeroshot_results.json` — metrics + total cost + `n_samples`

In [1]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI
from sklearn.metrics import classification_report, confusion_matrix, f1_score

LABELS: tuple[str, ...] = ("spam", "faq", "lead_intent", "escalate", "ambiguous")
DATA_CSV = Path("data/clinc150_mapped.csv")
RESULTS_PATH = Path("results/llm_zeroshot_results.json")

MODEL = "llama-3.1-8b-instant"
BASE_URL = "https://api.groq.com/openai/v1"
PRICE_INPUT_PER_1M = 0.05   # USD per 1M input tokens
PRICE_OUTPUT_PER_1M = 0.08  # USD per 1M output tokens

# Load .env from project root if the env var is not already set.
if not os.environ.get("GROQ_API_KEY"):
    try:
        from dotenv import load_dotenv
        load_dotenv("../.env")
    except ImportError:
        pass

api_key = os.environ.get("GROQ_API_KEY")
if not api_key:
    raise RuntimeError("GROQ_API_KEY not set. Add to .env or your environment.")
print(f"Using model: {MODEL} @ {BASE_URL}")
print(f"API key: {api_key[:6]}...{api_key[-4:]}")

Using model: llama-3.1-8b-instant @ https://api.groq.com/openai/v1
API key: gsk_yR...JhzU


In [2]:
df = pd.read_csv(DATA_CSV)
full_test = df[df["split"] == "test"].reset_index(drop=True)
print(f"Full test split: {len(full_test)} rows")

# Stratified sample: 100 per label. Using GroupBy.sample preserves the grouping
# column (pandas .apply strips it in modern versions).
SAMPLES_PER_LABEL = 100
test = (
    full_test.groupby("label", group_keys=False)
    .sample(n=SAMPLES_PER_LABEL, random_state=42)
    .reset_index(drop=True)
)
print(f"Sampled test set: {len(test)} rows")
print(test["label"].value_counts().reindex(list(LABELS)))

Full test split: 2290 rows
Sampled test set: 500 rows
label
spam           100
faq            100
lead_intent    100
escalate       100
ambiguous      100
Name: count, dtype: int64


In [3]:
SYSTEM_PROMPT = """You are a strict intent classifier for a business chat widget.
Read the user's message and respond with EXACTLY ONE of these 5 labels, lowercase, with no other text:

- spam: off-topic, gibberish, promotional spam, or out-of-scope for a business assistant
- faq: the user is asking for information (hours, prices, services, policies, locations)
- lead_intent: the user wants to buy, book, schedule, or be contacted by sales
- escalate: complaints, refunds, account issues, or explicit requests for a human
- ambiguous: very short or unclear messages that could fit multiple categories

Respond with ONLY the label name (spam | faq | lead_intent | escalate | ambiguous). No punctuation, no explanation, no quotes.
"""

def parse_label(raw: str) -> str:
    """Extract one of the 5 labels from the model's response; default to ambiguous."""
    s = raw.strip().lower().replace("\"", "").replace("'", "").strip(".,!?:; ")
    # exact match first
    if s in LABELS:
        return s
    # substring match (e.g. "the label is faq" or "faq.")
    for label in LABELS:
        if label in s:
            return label
    return "ambiguous"

# Quick parse sanity check.
for raw in ["faq", "FAQ", "  lead_intent  ", "spam.", "the answer is escalate", "unrelated"]:
    print(f"{raw!r:30s} -> {parse_label(raw)!r}")

'faq'                          -> 'faq'
'FAQ'                          -> 'faq'
'  lead_intent  '              -> 'lead_intent'
'spam.'                        -> 'spam'
'the answer is escalate'       -> 'escalate'
'unrelated'                    -> 'ambiguous'


In [4]:
client = OpenAI(api_key=api_key, base_url=BASE_URL)

# Groq free tier: 30 RPM hard cap for llama-3.1-8b-instant.
# Sleep 2.05s after each call (slight safety margin over 2.00s = 30/min).
PACING_S = 2.05

predictions: list[str] = []
latencies_ms: list[float] = []
tokens_in_total = 0
tokens_out_total = 0
fallback_count = 0
n = len(test)
loop_start = time.perf_counter()

for i, text in enumerate(test["text"].tolist()):
    call_start = time.perf_counter()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        temperature=0,
        max_tokens=10,
    )
    latencies_ms.append((time.perf_counter() - call_start) * 1000)
    raw = resp.choices[0].message.content or ""
    parsed = parse_label(raw)
    if raw.strip().lower() not in LABELS and parsed not in raw.lower():
        fallback_count += 1
    predictions.append(parsed)
    tokens_in_total += resp.usage.prompt_tokens
    tokens_out_total += resp.usage.completion_tokens
    if (i + 1) % 50 == 0 or i == n - 1:
        elapsed = time.perf_counter() - loop_start
        eta = (n - i - 1) * PACING_S  # pacing dominates
        print(f"  {i+1:>4}/{n}  elapsed={elapsed:6.1f}s  eta={eta:5.0f}s")
    # Pace AFTER the call (skip on last iteration).
    if i < n - 1:
        time.sleep(PACING_S)

total_elapsed = time.perf_counter() - loop_start
print(f"\nDone: {n} predictions in {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)")
print(f"Fallback parses (response did not contain any label): {fallback_count}")

    50/500  elapsed= 113.7s  eta=  922s


   100/500  elapsed= 225.5s  eta=  820s


   150/500  elapsed= 337.3s  eta=  717s


   200/500  elapsed= 451.5s  eta=  615s


   250/500  elapsed= 564.1s  eta=  512s


...

   350/500  elapsed= 786.9s  eta=  308s


   400/500  elapsed= 899.9s  eta=  205s


   450/500  elapsed=1013.9s  eta=  102s


   500/500  elapsed=1126.0s  eta=    0s



Done: 500 predictions in 1126.7s (18.8 min)
Fallback parses (response did not contain any label): 5


In [5]:
# (5) macro-F1 and per-class F1.
truths = test["label"].tolist()
test_macro_f1 = f1_score(truths, predictions, labels=list(LABELS), average="macro")
per_class_f1_arr = f1_score(truths, predictions, labels=list(LABELS), average=None)
per_class_f1 = {label: float(score) for label, score in zip(LABELS, per_class_f1_arr)}

print(f"Test macro-F1: {test_macro_f1:.4f}\n")
print("Per-class F1:")
for label, score in per_class_f1.items():
    print(f"  {label:12s} {score:.4f}")
print()
print(classification_report(truths, predictions, labels=list(LABELS), digits=4))

Test macro-F1: 0.4686

Per-class F1:
  spam         0.0577
  faq          0.4644
  lead_intent  0.5093
  escalate     0.6832
  ambiguous    0.6286

              precision    recall  f1-score   support

        spam     0.7500    0.0300    0.0577       100
         faq     0.3363    0.7500    0.4644       100
 lead_intent     0.6721    0.4100    0.5093       100
    escalate     0.6765    0.6900    0.6832       100
   ambiguous     0.6000    0.6600    0.6286       100

    accuracy                         0.5080       500
   macro avg     0.6070    0.5080    0.4686       500
weighted avg     0.6070    0.5080    0.4686       500



In [6]:
cm = confusion_matrix(truths, predictions, labels=list(LABELS))
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{lbl}" for lbl in LABELS],
    columns=[f"pred_{lbl}" for lbl in LABELS],
)
print(cm_df)

                  pred_spam  pred_faq  pred_lead_intent  pred_escalate  \
true_spam                 3        59                13              7   
true_faq                  1        75                 3              2   
true_lead_intent          0        41                41             13   
true_escalate             0        28                 1             69   
true_ambiguous            0        20                 3             11   

                  pred_ambiguous  
true_spam                     18  
true_faq                      19  
true_lead_intent               5  
true_escalate                  2  
true_ambiguous                66  


In [7]:
# (6) Mean latency per prediction.
mean_latency_ms = sum(latencies_ms) / len(latencies_ms)
p50 = sorted(latencies_ms)[len(latencies_ms) // 2]
p95 = sorted(latencies_ms)[int(len(latencies_ms) * 0.95)]
print(f"Latency: mean {mean_latency_ms:.1f} ms  p50 {p50:.1f} ms  p95 {p95:.1f} ms")

# (7) Cost.
input_cost = tokens_in_total * PRICE_INPUT_PER_1M / 1_000_000
output_cost = tokens_out_total * PRICE_OUTPUT_PER_1M / 1_000_000
total_cost_usd = input_cost + output_cost
cost_per_1k = total_cost_usd / (len(predictions) / 1000)

print(f"\nTokens:    input={tokens_in_total:,}  output={tokens_out_total:,}  total={tokens_in_total + tokens_out_total:,}")
print(f"Cost USD:  input=${input_cost:.4f}  output=${output_cost:.4f}  total=${total_cost_usd:.4f}")
print(f"Cost / 1k predictions: ${cost_per_1k:.4f}")

Latency: mean 205.6 ms  p50 159.1 ms  p95 462.7 ms

Tokens:    input=101,739  output=1,201  total=102,940
Cost USD:  input=$0.0051  output=$0.0001  total=$0.0052
Cost / 1k predictions: $0.0104


In [8]:
# (8) Results dict consumed by T134 (compare/export).
results = {
    "model": f"llm_zeroshot:{MODEL}",
    "macro_f1": float(test_macro_f1),
    "per_class_f1": per_class_f1,
    "latency_ms_per_prediction": float(mean_latency_ms),
    "cost_per_1k_predictions": float(cost_per_1k),
    "total_cost_usd": float(total_cost_usd),
    "tokens_used": {
        "input": int(tokens_in_total),
        "output": int(tokens_out_total),
        "total": int(tokens_in_total + tokens_out_total),
    },
    "fallback_parses": int(fallback_count),
    "n_samples": int(n),
    "sampling_note": "stratified 100/label from test split (random_state=42); spec asks for full 2290 but Groq free-tier 30 RPM made that impractical",
}
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.write_text(json.dumps(results, indent=2), encoding="utf-8")
print(json.dumps(results, indent=2))

{
  "model": "llm_zeroshot:llama-3.1-8b-instant",
  "macro_f1": 0.4686290216222952,
  "per_class_f1": {
    "spam": 0.057692307692307696,
    "faq": 0.46439628482972134,
    "lead_intent": 0.5093167701863354,
    "escalate": 0.6831683168316832,
    "ambiguous": 0.6285714285714286
  },
  "latency_ms_per_prediction": 205.6264122009161,
  "cost_per_1k_predictions": 0.010366060000000002,
  "total_cost_usd": 0.005183030000000001,
  "tokens_used": {
    "input": 101739,
    "output": 1201,
    "total": 102940
  },
  "fallback_parses": 5,
  "n_samples": 500,
  "sampling_note": "stratified 100/label from test split (random_state=42); spec asks for full 2290 but Groq free-tier 30 RPM made that impractical"
}


## Next

`05_compare_and_export.ipynb` (T134) reads `tfidf_logreg_results.json`,
`cnn_onnx_results.json`, and `llm_zeroshot_results.json`. It picks the
winner by macro-F1 with **latency / size / cost** tiebreakers (constitution
Principle IV prefers lean serving), copies the winning artifact into
`services/modelserver/artifacts/model.{onnx,joblib}`, and writes a
`model_card.yaml` with the deployment-choice rationale. The choice and the
comparison table also become entry #4 in `docs/DECISIONS.md` (T158).